# DP2 — Tracts covering the Deep Drilling Fields (DDF)

**Author:** dagoret  
**Date:** 2026-03-13  

This notebook:
1. Instantiates a Gen3 Butler on the DP0.2 repository
2. Loads the SkyMap and retrieves all available tracts
3. Identifies tracts that overlap the DDF fields (WFD and uDDF from DC2 / standard LSST DDFs)
4. Displays in Mollweide projection (astropy WCSAxes) the patches colored by tract, with DDF centers
5. Produces a ±5° Cartesian zoom plot for each individual DDF

**DDF reference:** `2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb`

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.path import Path
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
import seaborn as sns

# Astropy
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.wcs import WCS

# LSST Stack
import lsst.daf.butler as dafButler
import lsst.geom as geom
import lsst.sphgeom as sphgeom

print(f"Matplotlib version : {matplotlib.__version__}")
print(f"Seaborn version    : {sns.__version__}")
print("Imports OK")

In [ ]:
#import ipywidgets as widgets
#%matplotlib widget

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG   = 'DP2_DDF_SKYMAP01'
DIR_DATA = f'data_{NB_TAG}'
DIR_FIGS = f'figs_{NB_TAG}'
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f'Data : {os.path.abspath(DIR_DATA)}')
print(f'Figs : {os.path.abspath(DIR_FIGS)}')

## 1. Definition of the Deep Drilling Fields

Coordinates from `2026-03-11_DP2_SurveyPropertyMaps_AllBands_AllDDF.ipynb`.

For DP0.2 (based on DC2), the simulation covers ~300 deg² centered around RA≈61.9°, Dec≈-35.8°.  
The DC2 ultra-deep drilling field (uDDF) is centered on RA=55.064°, Dec=-29.783°.  
The standard LSST DDF fields are also listed below.

In [ ]:
# -------------------------------------------------------------------------
# DDF dictionary: name -> (RA_deg, Dec_deg)
# Source: reference notebook 2026-03-11 + LSST literature
# -------------------------------------------------------------------------
DDF_COORDS = {
    # --- Fields present in DC2 / DP0.2 ---
    #"DC2_WFD_center" : (61.863, -35.790),   # Center of the DC2 WFD simulation
    #"DC2_uDDF"       : (55.064, -29.783),   # DC2 ultra-deep drilling field

    # --- Official LSST DDFs (outside the DC2 footprint, for reference) ---
    "COSMOS"         : (150.119, +2.206),
    "ECDFS"          : (53.125, -28.100),   # Extended Chandra Deep Field South
    "ELAIS-S1"       : (9.450,  -44.000),
    "XMM-LSS"        : (35.708, -4.750),
    "EDFS-a"         : (58.900, -49.315),   # Euclid Deep Field South (a)
    "EDFS-b"         : (63.600, -47.600),   # Euclid Deep Field South (b)
    "EDFS"           : (61.24,-48.423),
    "M49"            : (187.4 ,8.0),
}

# Search radius around each DDF (in degrees)
# An HSC-like tract is ~1.7 deg on a side -> radius of 1.5 deg captures the central tract(s)
SEARCH_RADIUS_DEG = 1.8

print("DDF fields defined:")
for name, (ra, dec) in DDF_COORDS.items():
    print(f"  {name:20s}  RA={ra:8.3f}°   Dec={dec:+8.3f}°")

## 2. Butler instantiation and SkyMap loading

In [ ]:
# -------------------------------------------------------------------------
# Butler DP0.2 configuration
# Adapt REPO path according to the environment (RSP, NERSC, local...)
# -------------------------------------------------------------------------
REPO       = "dp2_prep"                          # RSP alias
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"   


butler  = dafButler.Butler(REPO, collections=COLLECTION)
skymap  = butler.get("skyMap", skymap=SKYMAP_NAME)

n_tracts = len(skymap)
print(f"Butler instantiated : {REPO}")
print(f"Collection          : {COLLECTION}")
print(f"SkyMap              : {SKYMAP_NAME}")
print(f"Number of tracts    : {n_tracts}")

## 3. Finding tracts that overlap the DDFs

In [ ]:
# -------------------------------------------------------------------------
# Function: returns all tracts whose footprint covers a disk (ra, dec, radius)
#
# Approach: grid of points sampling the search disk.
# For each point we call skymap.findTract() which always returns
# a TractInfo (stable method across all LSST stack versions).
# We take the union of found IDs -> deduplicated.
#
# This approach entirely avoids sphgeom.Circle / poly.intersects()
# whose return type (bool vs Relationship) varies across versions.
# -------------------------------------------------------------------------
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=SEARCH_RADIUS_DEG):
    """
    Returns the deduplicated list of TractInfo whose footprint overlaps
    the disk (ra_deg, dec_deg, radius_deg).
    Uses only skymap.findTract(SpherePoint) — stable API.
    """
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)   # avoid division by zero at poles
    step    = 0.35   # grid step in degrees (< tract size)

    found_ids = set()

    # Rectangular grid covering the disk
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            # Angular distance to center (local spherical approximation)
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s  = ra_deg  + dra / cos_dec    # projection correction
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = geom.SpherePoint(ra_s * geom.degrees, dec_s * geom.degrees)
            try:
                ti = skymap.findTract(sp)
                found_ids.add(ti.tract_id)
            except Exception:
                pass

    return [skymap[tid] for tid in sorted(found_ids)]


# -------------------------------------------------------------------------
# Search for each DDF
# -------------------------------------------------------------------------
# ddf_tracts: dict  ddf_name -> list of TractInfo
ddf_tracts = {}

for ddf_name, (ra, dec) in DDF_COORDS.items():
    tracts = find_tracts_for_coord(skymap, ra, dec, radius_deg=SEARCH_RADIUS_DEG)
    ddf_tracts[ddf_name] = tracts
    ids = [t.tract_id for t in tracts]
    print(f"{ddf_name:20s}  -> {len(tracts)} tract(s): {ids}")

In [ ]:
# -------------------------------------------------------------------------
# Build the deduplicated list of all selected tracts
# with their DDF association(s)
# -------------------------------------------------------------------------
tract_to_ddfs = {}   # tract_id -> [ddf_name, ...]

for ddf_name, tracts in ddf_tracts.items():
    for t in tracts:
        tid = t.tract_id
        tract_to_ddfs.setdefault(tid, [])
        if ddf_name not in tract_to_ddfs[tid]:
            tract_to_ddfs[tid].append(ddf_name)

# Unique index of selected TractInfo
selected_tract_ids   = sorted(tract_to_ddfs.keys())
selected_tract_infos = {tid: skymap[tid] for tid in selected_tract_ids}

print(f"\nTotal selected tracts (deduplicated): {len(selected_tract_ids)}")
print(f"Tract IDs: {selected_tract_ids}")

## 4. Geometry utilities: extracting patch corners

For each selected tract we extract the RA/Dec coordinates of the corners of each patch (inner bbox -> WCS -> sky coords).

In [ ]:
def get_tract_corners_radec(tract_info):
    """
    Returns the 4 corners of the tract (outer sky polygon) in (RA, Dec) degrees.
    Order is preserved to form a closed polygon.
    """
    vertices = tract_info.getOuterSkyPolygon().getVertices()
    corners = []
    for v in vertices:
        sp = geom.SpherePoint(v)
        corners.append((sp.getRa().asDegrees(), sp.getDec().asDegrees()))
    return np.array(corners)


def get_patch_corners_radec(tract_info, patch_info):
    """
    Returns the 4 corners of a patch (inner bbox) in RA/Dec coordinates in degrees.
    """
    wcs  = tract_info.getWcs()
    bbox = patch_info.getInnerBBox()

    pixel_corners = [
        geom.Point2D(bbox.getMinX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMinY()),
        geom.Point2D(bbox.getMaxX(), bbox.getMaxY()),
        geom.Point2D(bbox.getMinX(), bbox.getMaxY()),
    ]
    radec = []
    for pix in pixel_corners:
        sp = wcs.pixelToSky(pix)
        radec.append((sp.getRa().asDegrees(), sp.getDec().asDegrees()))
    return np.array(radec)


def wrap_ra(ra_arr, ref_ra=180.0):
    """
    Wraps RA into [-180, 180] for the astropy Mollweide projection.
    ra_arr in degrees -> returns degrees centered around ref_ra.
    """
    ra = np.array(ra_arr, dtype=float)
    ra = np.where(ra > 180.0, ra - 360.0, ra)
    return ra


print("Utility functions defined.")

In [ ]:
# -------------------------------------------------------------------------
# Pre-compute all patch polygons for the selected tracts
# -------------------------------------------------------------------------
# Structure: patch_polygons[tract_id] = list of (ra_arr, dec_arr) [5 pts, closed polygon]

patch_polygons = {}   # tract_id -> list of np.ndarray shape (5, 2) = [(ra, dec), ...]
tract_centers  = {}   # tract_id -> (ra_center, dec_center)

for tid in selected_tract_ids:
    ti = selected_tract_infos[tid]
    polys = []
    for patch_info in ti:
        corners = get_patch_corners_radec(ti, patch_info)   # shape (4, 2)
        # Close the polygon
        closed  = np.vstack([corners, corners[0]])           # shape (5, 2)
        polys.append(closed)

    patch_polygons[tid] = polys

    # Tract center (barycenter of tract corners)
    tc = get_tract_corners_radec(ti)
    ra_c  = np.mean(tc[:, 0])
    dec_c = np.mean(tc[:, 1])
    tract_centers[tid] = (ra_c, dec_c)

    n_patches = sum(1 for _ in ti)
    print(f"Tract {tid:5d}  center=({ra_c:.2f}°, {dec_c:+.2f}°)  "
          f"patches={n_patches}  DDFs={tract_to_ddfs[tid]}")

## 5. Visualization — Mollweide Projection (astropy WCSAxes)

- **Patches** are colored by tract, with a distinct color per tract.
- **DDF centers** are marked with a star symbol.
- The **legend** is placed to the left, outside the image area.
- Seaborn palette for well-distinct colors.

In [ ]:
# =========================================================================
# Graphic parameters
# =========================================================================

# --- Well-distinct seaborn color palette ---
n_tracts_sel = len(selected_tract_ids)

# For <= 10 tracts: qualitative palette; beyond that use hls
if n_tracts_sel <= 10:
    palette = sns.color_palette("tab10", n_tracts_sel)
elif n_tracts_sel <= 20:
    palette = sns.color_palette("tab20", n_tracts_sel)
else:
    palette = sns.color_palette("hls", n_tracts_sel)

tract_color = {tid: palette[i] for i, tid in enumerate(selected_tract_ids)}

# --- Symbols and colors for DDF centers ---
DDF_MARKER      = "+"
DDF_MARKERSIZE  = 18
DDF_EDGECOLOR   = "black"
DDF_FACECOLOR   = "gold"


# =========================================================================
# Create the Mollweide figure with matplotlib
# Note: matplotlib natively supports the 'mollweide' projection
#       with angles in RADIANS and x-axis in [-pi, pi]
# =========================================================================

fig = plt.figure(figsize=(18, 9))

# Mollweide axes (matplotlib built-in — handles the spherical projection)
# Reserve space on the left for the legend
ax = fig.add_subplot(111, projection='mollweide')
ax.set_facecolor('#f0f4f8')   # slightly grey background

ax.grid(True, color='black', linewidth=0.6, alpha=0.8)

# =========================================================================
# Plot patches colored by tract
# =========================================================================

legend_handles = []

for tid in selected_tract_ids:
    color   = tract_color[tid]
    ddfs    = tract_to_ddfs[tid]
    label   = f"Tract {tid} :: " + ", ".join(ddfs)

    patch_added = False
    for poly in patch_polygons[tid]:
        ra_deg  = poly[:, 0]   # (5,)
        dec_deg = poly[:, 1]   # (5,)

        # Convert to radians for matplotlib Mollweide
        # RA: astropy Mollweide convention -> east on the left, so -RA
        ra_rad  = -np.deg2rad(wrap_ra(ra_deg))
        dec_rad =  np.deg2rad(dec_deg)

        # Check for phase discontinuity (wrap)
        # (case of polygons crossing RA=0/360 -- absent in DC2)
        if np.any(np.abs(np.diff(ra_rad)) > np.pi / 2):
            continue   # polygon cut by the edge -- skip for simplicity

        verts  = np.column_stack([ra_rad, dec_rad])
        patch  = mpatches.Polygon(
            verts[:-1],          # without the closing point
            closed=True,
            facecolor=color,
            edgecolor='white',
            linewidth=0.3,
            alpha=0.65,
            transform=ax.transData,
        )
        ax.add_patch(patch)
        patch_added = True

    if patch_added:
        legend_handles.append(
            mpatches.Patch(facecolor=color, edgecolor='grey',
                           linewidth=0.8, alpha=0.85, label=label)
        )

# =========================================================================
# Plot tract outlines (thick border)
# =========================================================================

for tid in selected_tract_ids:
    color = tract_color[tid]
    ti    = selected_tract_infos[tid]
    tc    = get_tract_corners_radec(ti)

    ra_wrap = wrap_ra(tc[:, 0])
    ra_rad  = -np.deg2rad(np.append(ra_wrap, ra_wrap[0]))
    dec_rad =  np.deg2rad(np.append(tc[:, 1], tc[0, 1]))

    ax.plot(ra_rad, dec_rad,
            color=color, linewidth=1.8,
            solid_capstyle='round', zorder=3,
            path_effects=[pe.Stroke(linewidth=3.5, foreground='black'), pe.Normal()])

# =========================================================================
# Plot DDF centers
# =========================================================================

ddf_handles = []
for ddf_name, (ra, dec) in DDF_COORDS.items():
    ra_rad  = -np.deg2rad(wrap_ra(ra))
    dec_rad =  np.deg2rad(dec)
    ax.plot(ra_rad, dec_rad,
            marker=DDF_MARKER,
            markersize=DDF_MARKERSIZE,
            color=DDF_FACECOLOR,
            markeredgecolor=DDF_EDGECOLOR,
            markeredgewidth=1.0,
            zorder=10,
            linestyle='none')
    # Annotation
    ax.annotate(
        ddf_name,
        xy=(ra_rad, dec_rad),
        xytext=(ra_rad + np.deg2rad(2.0), dec_rad + np.deg2rad(1.5)),
        fontsize=12,
        fontweight='bold',
        color='black',
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
        zorder=11,
    )

ddf_handles.append(
    plt.Line2D([0], [0],
               marker=DDF_MARKER, linestyle='none',
               markersize=12,
               color=DDF_FACECOLOR,
               markeredgecolor=DDF_EDGECOLOR,
               markeredgewidth=1.0,
               label='DDF center')
)

# =========================================================================
# Legend — outside to the left
# =========================================================================

all_handles = legend_handles + ddf_handles

leg = ax.legend(
    handles=all_handles,
    loc='center left',
    bbox_to_anchor=(-0.30, 0.50),
    fontsize=8,
    frameon=True,
    framealpha=0.92,
    edgecolor='grey',
    title='Tract :: DDF',
    title_fontsize=9,
    ncol=1,
)

# =========================================================================
# Axes and title
# =========================================================================

# RA ticks in degrees (matplotlib Mollweide uses radians)
tick_ra_deg  = np.arange(0, 360, 30)
tick_ra_rad  = -np.deg2rad(wrap_ra(tick_ra_deg))
ax.set_xticks(tick_ra_rad)
ax.set_xticklabels(
    [f"{r:.0f}°" for r in tick_ra_deg],
    fontsize=16
)

tick_dec_deg = np.arange(-75, 90, 15)
ax.set_yticks(np.deg2rad(tick_dec_deg))
ax.set_yticklabels([f"{d:+.0f}°" for d in tick_dec_deg], fontsize=12)

ax.set_xlabel("Right Ascension (J2000)", fontsize=12, labelpad=12)
ax.set_ylabel("Declination (J2000)", fontsize=12)

ax.set_title(
    "DP2 — SkyMap DP2 Tracts covering the Deep Drilling Fields\n"
    "(Mollweide projection — patches colored by tract)",
    fontsize=13, fontweight='bold', pad=14
)

plt.tight_layout()
fig.subplots_adjust(left=0.28)   # space for the legend on the left

# Save
outfile = "DDF_Tracts_Mollweide_DP2.png"
plt.savefig(outfile, dpi=180, bbox_inches='tight')
print(f"Figure saved: {outfile}")
plt.show()

## 6. Summary table

In [ ]:
import pandas as pd

rows = []
for tid in selected_tract_ids:
    ra_c, dec_c = tract_centers[tid]
    ddfs        = ", ".join(tract_to_ddfs[tid])
    n_p         = len(patch_polygons[tid])
    rows.append({
        "tract_id"    : tid,
        "RA_center"   : round(ra_c, 3),
        "Dec_center"  : round(dec_c, 3),
        "n_patches"   : n_p,
        "DDF(s)"      : ddfs,
    })

df = pd.DataFrame(rows).set_index("tract_id")
print(f"Selected tracts ({len(df)} total):")
display(df)

## 7. Per-DDF zoom plots (±5° Cartesian view)

For each DDF, a ±5° Cartesian zoom (RA/Dec) is produced showing:
- All tracts associated with that DDF, with patches colored by tract ID
- The DDF center marked with a gold star
- A legend outside to the left listing tract ID :: DDF name

The zoom half-width is set to `ZOOM_HALF_DEG = 5.0` degrees (adjustable).

In [ ]:
# =========================================================================
# Helper function: Cartesian ±N° zoom plot for a single DDF
# =========================================================================

# Half-width of the zoom window in degrees (same in RA and Dec)
ZOOM_HALF_DEG = 5.0

# Star marker for DDF centers in zoom plots
ZOOM_DDF_MARKER     = "*"
ZOOM_DDF_MARKERSIZE = 20
ZOOM_DDF_FACECOLOR  = "gold"
ZOOM_DDF_EDGECOLOR  = "black"


def plot_ddf_zoom(ddf_name, ra_center, dec_center,
                  half_deg=ZOOM_HALF_DEG,
                  all_ddf_coords=None,
                  save=True):
    """
    Produce a Cartesian ±half_deg zoom plot centred on (ra_center, dec_center)
    for the given DDF.

    Parameters
    ----------
    ddf_name     : str   — name of the target DDF (used for title and filename)
    ra_center    : float — right ascension of the DDF center (degrees)
    dec_center   : float — declination of the DDF center (degrees)
    half_deg     : float — half-width of the zoom window (degrees)
    all_ddf_coords : dict or None — if provided, all other DDF centers are
                   overplotted as small grey crosses for spatial context
    save         : bool  — if True, save the figure to a PNG file
    """
    # RA window (note: RA is inverted on the x-axis by convention)
    cos_dec = max(np.cos(np.deg2rad(dec_center)), 0.01)
    ra_half = half_deg / cos_dec          # stretch window in RA to account for projection
    RA_MIN  = ra_center - ra_half
    RA_MAX  = ra_center + ra_half
    DEC_MIN = dec_center - half_deg
    DEC_MAX = dec_center + half_deg

    fig, ax = plt.subplots(figsize=(9, 8))
    ax.set_facecolor('#f0f4f8')            # light grey background
    ax.invert_xaxis()                      # RA increases towards the left

    legend_handles = []

    # -----------------------------------------------------------------
    # Draw patches for all tracts associated with this DDF
    # -----------------------------------------------------------------
    tracts_for_this_ddf = ddf_tracts.get(ddf_name, [])
    local_tract_ids = sorted(t.tract_id for t in tracts_for_this_ddf)

    for tid in local_tract_ids:
        color = tract_color[tid]
        ddfs  = tract_to_ddfs[tid]
        label = f"Tract {tid} :: " + ", ".join(ddfs)

        patch_added = False
        for poly in patch_polygons[tid]:
            ra_deg  = poly[:-1, 0]    # 4 vertices (open polygon)
            dec_deg = poly[:-1, 1]

            # Only draw patches that overlap the zoom window
            if (ra_deg.max() < RA_MIN or ra_deg.min() > RA_MAX or
                    dec_deg.max() < DEC_MIN or dec_deg.min() > DEC_MAX):
                continue

            p = mpatches.Polygon(
                list(zip(ra_deg, dec_deg)),
                closed=True,
                facecolor=color, edgecolor='white',
                linewidth=0.3, alpha=0.65,
            )
            ax.add_patch(p)
            patch_added = True

        if patch_added:
            # Draw the tract outline with a thick colored border
            ti = selected_tract_infos[tid]
            tc = get_tract_corners_radec(ti)
            ax.plot(np.append(tc[:, 0], tc[0, 0]),
                    np.append(tc[:, 1], tc[0, 1]),
                    color=color, linewidth=2.0, zorder=3,
                    path_effects=[
                        pe.Stroke(linewidth=3.5, foreground='black'),
                        pe.Normal()
                    ])
            # Add tract ID annotation at the tract center
            ra_c, dec_c = tract_centers[tid]
            ax.text(ra_c, dec_c, str(tid),
                    fontsize=8, ha='center', va='center',
                    color='black', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='white', alpha=0.6, edgecolor='none'),
                    zorder=5)
            legend_handles.append(
                mpatches.Patch(facecolor=color, edgecolor='grey',
                               linewidth=0.8, alpha=0.85, label=label)
            )

    # -----------------------------------------------------------------
    # Plot the target DDF center (gold star)
    # -----------------------------------------------------------------
    ax.plot(ra_center, dec_center,
            marker=ZOOM_DDF_MARKER,
            markersize=ZOOM_DDF_MARKERSIZE,
            color=ZOOM_DDF_FACECOLOR,
            markeredgecolor=ZOOM_DDF_EDGECOLOR,
            markeredgewidth=1.2,
            zorder=10, linestyle='none')
    ax.annotate(
        ddf_name,
        xy=(ra_center, dec_center),
        xytext=(ra_center + 0.4 * ra_half / half_deg,
                dec_center + 0.4),
        fontsize=11, fontweight='bold', color='black',
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
        zorder=11,
    )

    # -----------------------------------------------------------------
    # Overplot other DDF centers as small grey crosses for context
    # -----------------------------------------------------------------
    if all_ddf_coords is not None:
        for other_name, (other_ra, other_dec) in all_ddf_coords.items():
            if other_name == ddf_name:
                continue
            # Only draw if it falls within the zoom window
            if (RA_MIN <= other_ra <= RA_MAX and
                    DEC_MIN <= other_dec <= DEC_MAX):
                ax.plot(other_ra, other_dec,
                        marker='+', markersize=10,
                        color='dimgrey', markeredgewidth=1.5,
                        zorder=9, linestyle='none')
                ax.text(other_ra, other_dec + 0.15, other_name,
                        fontsize=7, color='dimgrey', ha='center')

    # -----------------------------------------------------------------
    # Legend entry for DDF center marker
    # -----------------------------------------------------------------
    legend_handles.append(
        plt.Line2D([0], [0],
                   marker=ZOOM_DDF_MARKER, linestyle='none',
                   markersize=12,
                   color=ZOOM_DDF_FACECOLOR,
                   markeredgecolor=ZOOM_DDF_EDGECOLOR,
                   markeredgewidth=1.0,
                   label=f'{ddf_name} center')
    )

    # -----------------------------------------------------------------
    # Axes limits, labels, grid
    # -----------------------------------------------------------------
    ax.set_xlim(RA_MAX, RA_MIN)     # RA inverted
    ax.set_ylim(DEC_MIN, DEC_MAX)
    ax.set_xlabel("Right Ascension (deg)", fontsize=12)
    ax.set_ylabel("Declination (deg)", fontsize=12)
    ax.set_title(
        f"DP2 — {ddf_name} : tracts and patches\n"
        f"RA ∈ [{RA_MIN:.2f}°, {RA_MAX:.2f}°]   "
        f"Dec ∈ [{DEC_MIN:.1f}°, {DEC_MAX:.1f}°]   "
        f"(±{half_deg}°)",
        fontsize=12, fontweight='bold'
    )
    ax.grid(True, alpha=0.4, linestyle='--', color='grey')

    # -----------------------------------------------------------------
    # Legend — outside to the left
    # -----------------------------------------------------------------
    ax.legend(
        handles=legend_handles,
        loc='center left',
        bbox_to_anchor=(-0.35, 0.50),
        fontsize=8.5,
        frameon=True, framealpha=0.92, edgecolor='grey',
        title='Tract :: DDF', title_fontsize=9,
    )

    plt.tight_layout()
    fig.subplots_adjust(left=0.30)   # extra space for the legend

    if save:
        fname = f"DDF_Zoom_{ddf_name.replace('-', '_')}_DP2.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        print(f"  Saved: {fname}")

    plt.show()
    plt.close(fig)


print("Zoom helper function defined.")

In [ ]:
# =========================================================================
# Loop over all DDFs and produce a ±5° zoom plot for each one
# =========================================================================

print(f"Producing ±{ZOOM_HALF_DEG}° zoom plots for {len(DDF_COORDS)} DDFs...\n")

for ddf_name, (ra_center, dec_center) in DDF_COORDS.items():
    print(f"--- {ddf_name}  (RA={ra_center:.3f}°, Dec={dec_center:+.3f}°) ---")
    tracts_here = ddf_tracts.get(ddf_name, [])
    if not tracts_here:
        print(f"  No tracts found within {SEARCH_RADIUS_DEG}° — skipping zoom plot.")
        continue
    print(f"  Tracts: {[t.tract_id for t in tracts_here]}")
    plot_ddf_zoom(
        ddf_name    = ddf_name,
        ra_center   = ra_center,
        dec_center  = dec_center,
        half_deg    = ZOOM_HALF_DEG,
        all_ddf_coords = DDF_COORDS,   # show other DDFs if they fall in the window
        save        = True,
    )

print("\nAll zoom plots done.")

---
## Technical notes

| Parameter | Value |
|---|---|
| Butler repository | `dp2_prep` |
| Collection | `LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545` |
| SkyMap | `lsst_cells_v2` |
| Search radius | 1.8° around each DDF center |
| Global projection | Mollweide (native matplotlib) |
| Per-DDF zoom | Cartesian ±5° (adjustable via `ZOOM_HALF_DEG`) |
| Color palette | `seaborn tab10/tab20/hls` depending on number of tracts |

**Adapt** `REPO`, `COLLECTION` and `SKYMAP_NAME` if working in a different environment than the RSP (NERSC, local stack, etc.).